# Logistic Regression

## 1. Load the training data

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, ParameterGrid, StratifiedKFold
from sklearn.pipeline import Pipeline

# Find the data and preprocessing code from either notebook location.
working_dir = Path.cwd()
project_root = next(
    (path for path in (working_dir, *working_dir.parents)
     if (path / 'src' / 'preprocessing_pipeline.py').is_file()
     and (path / 'Dataset' / 'processed').is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Run this notebook from the project root or a subfolder.')

sys.path.insert(0, str(project_root))
from src.preprocessing_pipeline import TARGET, build_preprocessor

RANDOM_STATE = 42
BAD, GOOD = 0, 1
CLASS_NAMES = ['bad', 'good']
data_dir = project_root / 'Dataset' / 'processed'

In [2]:
X_train = pd.read_csv(data_dir / 'X_train.csv')
y_train = pd.read_csv(data_dir / 'y_train.csv')[TARGET]

print(f'Training set has {len(X_train)} rows and {X_train.shape[1]} features.')
display(y_train.value_counts().reindex([BAD, GOOD]).set_axis(CLASS_NAMES).to_frame('count'))

Training set has 700 rows and 21 features.


,count
bad,210
good,490


## 2. Tune with five stratified folds

Run the first grid, then test more settings around the better results. Use bad-class F1 to rank the settings.

### Set up the first grid

In [3]:
pipeline = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('model', LogisticRegression(max_iter=6000, random_state=RANDOM_STATE)),
])

class_weights = [
    None, 'balanced', {BAD: 1.5, GOOD: 1}, {BAD: 2, GOOD: 1},
    {BAD: 2.75, GOOD: 1}, {BAD: 3, GOOD: 1}, {BAD: 4, GOOD: 1},
]
param_grid = [
    {
        'model__solver': ['liblinear'],
        'model__C': [0.003, 0.01, 0.02, 0.03, 0.05, 0.07, 0.08, 0.1, 0.15,
                     0.22, 0.33, 0.5, 0.75, 1.0, 2.0, 3.0, 10.0, 100.0],
        'model__l1_ratio': [0.0, 1.0],
        'model__class_weight': class_weights,
    },
    {
        'model__solver': ['saga'],
        'model__C': [0.03, 0.08, 0.15, 0.3, 0.75, 1.0, 3.0],
        'model__l1_ratio': [0.25, 0.5, 0.75],
        'model__class_weight': class_weights,
    },
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'f1': make_scorer(f1_score, pos_label=BAD, zero_division=0),
    'recall': make_scorer(recall_score, pos_label=BAD, zero_division=0),
    'precision': make_scorer(precision_score, pos_label=BAD, zero_division=0),
    'accuracy': 'accuracy',
}
ranking_columns = [f'mean_test_{metric}' for metric in scoring]

In [4]:
search = GridSearchCV(
    pipeline, param_grid=param_grid, scoring=scoring, cv=cv,
    refit=False, error_score='raise',
)
search.fit(X_train, y_train)
cv_results = pd.DataFrame(search.cv_results_)

prior_best = cv_results.sort_values(ranking_columns, ascending=False, kind='stable').iloc[0]
print(f"Prior search best mean validation F1 (bad) = {prior_best['mean_test_f1']:.4f}", flush=True)

Prior search best mean validation F1 (bad) = 0.6133


### Try nearby settings

In [5]:
refinement_grid = [
    {
        'model__solver': ['liblinear'],
        'model__l1_ratio': [0.0],
        'model__C': [0.055, 0.06, 0.065, 0.0725, 0.075, 0.0775, 0.0825, 0.085, 0.09,
                     0.095, 0.105, 0.12, 0.135, 0.165, 0.18],
        'model__class_weight': [
            'balanced',
            *[{BAD: weight, GOOD: 1} for weight in
              [1.7, 1.8, 1.9, 1.95, 2.0, 2.05, 2.1, 2.2, 2.33, 2.45, 2.55]],
        ],
    },
    {
        'model__solver': ['saga'],
        'model__C': [0.12, 0.2, 0.3, 0.45, 0.6],
        'model__l1_ratio': [0.1, 0.2, 0.35, 0.6, 0.85],
        'model__class_weight': ['balanced', {BAD: 1.9, GOOD: 1}, {BAD: 2.2, GOOD: 1}],
    },
    {
        'model__solver': ['liblinear'],
        'model__l1_ratio': [0.0],
        'model__C': [0.06, 0.08, 0.11],
        'model__class_weight': ['balanced', {BAD: 1.8, GOOD: 1}, {BAD: 2.0, GOOD: 1},
                                {BAD: 2.2, GOOD: 1}],
        'model__intercept_scaling': [0.5, 2.0, 5.0],
    },
]

In [6]:
# Skip settings already tested in the first search.
previous_candidates = cv_results['params'].tolist()
new_candidates = [
    candidate for candidate in ParameterGrid(refinement_grid)
    if candidate not in previous_candidates
]
candidate_grid = [
    {param: [value] for param, value in candidate.items()} for candidate in new_candidates
]
print(f'Validating {len(new_candidates)} new candidates across five folds.', flush=True)
focused_search = GridSearchCV(
    pipeline, candidate_grid, scoring=scoring, cv=cv,
    refit=False, error_score='raise',
)
focused_search.fit(X_train, y_train)
focused_results = pd.DataFrame(focused_search.cv_results_)
print(f'Evaluated {len(focused_results)} additional configurations.', flush=True)
cv_results = pd.concat([cv_results, focused_results], ignore_index=True)

Validating 291 new candidates across five folds.
Evaluated 291 additional configurations.


### Pick the best settings

In [7]:
ranked_results = cv_results.sort_values(ranking_columns, ascending=False, kind='stable')
best_result = ranked_results.iloc[0]
best_params = best_result['params']

print(f'Evaluated {len(cv_results)} candidates across five folds.')
print(f"Best mean validation F1 (bad) = {best_result['mean_test_f1']:.4f}")
display(pd.Series(best_params, name='Selected value', dtype=object).to_frame())

Evaluated 690 candidates across five folds.
Best mean validation F1 (bad) = 0.6145


,Selected value
model__C,0.0825
model__class_weight,"{0: 2.0, 1: 1}"
model__l1_ratio,0.0
model__solver,liblinear


In [8]:
metric_columns = {
    'mean_test_f1': 'F1-score (bad)',
    'mean_test_recall': 'Recall (bad)',
    'mean_test_precision': 'Precision (bad)',
    'mean_test_accuracy': 'Accuracy',
}
candidate_columns = {**metric_columns, 'std_test_f1': 'F1 SD', 'params': 'Parameters'}

print('Top five settings by validation F1')
with pd.option_context('display.max_colwidth', None):
    display(
        ranked_results.head(5)[list(candidate_columns)]
        .rename(columns=candidate_columns)
        .round(4)
        .reset_index(drop=True)
    )

Top five settings by validation F1


,F1-score (bad),Recall (bad),Precision (bad),Accuracy,F1 SD,Parameters
0,0.6145,0.7000,0.5491,0.7371,0.0321,"{'model__C': 0.0825, 'model__class_weight': {0: 2.0, 1: 1}, 'model__l1_ratio': 0.0, 'model__solver': 'liblinear'}"
1,0.6144,0.7143,0.5407,0.7314,0.0301,"{'model__C': 0.055, 'model__class_weight': {0: 2.1, 1: 1}, 'model__l1_ratio': 0.0, 'model__solver': 'liblinear'}"
2,0.6141,0.6952,0.5515,0.7386,0.0386,"{'model__C': 0.0775, 'model__class_weight': {0: 1.95, 1: 1}, 'model__l1_ratio': 0.0, 'model__solver': 'liblinear'}"
3,0.6140,0.7286,0.5320,0.7257,0.0270,"{'model__C': 0.09, 'model__class_weight': {0: 2.2, 1: 1}, 'model__l1_ratio': 0.0, 'model__solver': 'liblinear'}"
4,0.6133,0.7000,0.5473,0.7357,0.0335,"{'model__C': 0.08, 'model__class_weight': {0: 2, 1: 1}, 'model__l1_ratio': 0.0, 'model__solver': 'liblinear'}"


In [9]:
# If scores tie, show the setting ranked higher by F1.
metric_leaders = pd.DataFrame([
    ranked_results.loc[ranked_results[column].idxmax(), [*metric_columns, 'params']]
    for column in metric_columns
], index=pd.Index(metric_columns.values(), name='Metric maximized'))
print('Validation leaders by metric')
with pd.option_context('display.max_colwidth', None):
    display(metric_leaders.rename(columns=candidate_columns).round(4))

Validation leaders by metric


,F1-score (bad),Recall (bad),Precision (bad),Accuracy,Parameters
Metric maximized,,,,,
F1-score (bad),0.6145,0.7000,0.5491,0.7371,"{'model__C': 0.0825, 'model__class_weight': {0: 2.0, 1: 1}, 'model__l1_ratio': 0.0, 'model__solver': 'liblinear'}"
Recall (bad),0.4615,1.0000,0.3000,0.3000,"{'model__C': 0.003, 'model__class_weight': None, 'model__l1_ratio': 1.0, 'model__solver': 'liblinear'}"
Precision (bad),0.1048,0.0571,0.7733,0.7100,"{'model__C': 0.03, 'model__class_weight': None, 'model__l1_ratio': 1.0, 'model__solver': 'liblinear'}"
Accuracy,0.5307,0.4667,0.6177,0.7543,"{'model__C': 0.5, 'model__class_weight': None, 'model__l1_ratio': 0.0, 'model__solver': 'liblinear'}"


In [10]:
# Read the scores already saved by the search for each fold.
fold_rows = []
for fold, (fit_rows, validation_rows) in enumerate(cv.split(X_train, y_train), start=1):
    fold_scores = {
        metric_columns[f'mean_test_{metric}']: best_result[f'split{fold - 1}_test_{metric}']
        for metric in scoring
    }
    fold_rows.append({
        'Fold': fold,
        'Fitting rows': len(fit_rows),
        'Validation rows': len(validation_rows),
        'Validation bad cases': int(y_train.iloc[validation_rows].eq(BAD).sum()),
        **fold_scores,
    })
print('Validation scores by fold for the selected setting')
display(pd.DataFrame(fold_rows).set_index('Fold').round(4))

Validation scores by fold for the selected setting


,Fitting rows,Validation rows,Validation bad cases,F1-score (bad),Recall (bad),Precision (bad),Accuracy
Fold,,,,,,,
1,560,140,42,0.5806,0.6429,0.5294,0.7214
2,560,140,42,0.6602,0.8095,0.5574,0.7500
3,560,140,42,0.5773,0.6667,0.5091,0.7071
4,560,140,42,0.6374,0.6905,0.5918,0.7643
5,560,140,42,0.6170,0.6905,0.5577,0.7429


### Fit on the full training set

In [11]:
best_model = pipeline.set_params(**best_params).fit(X_train, y_train)

## 3. Evaluate on the held-out test set

In [12]:
X_test = pd.read_csv(data_dir / 'X_test.csv')
y_test = pd.read_csv(data_dir / 'y_test.csv')[TARGET]

y_pred = best_model.predict(X_test)

In [13]:
matrix = confusion_matrix(y_test, y_pred, labels=[BAD, GOOD])
(tp, fn), (fp, tn) = matrix

display(pd.DataFrame(
    [[f'{tp} (TP)', f'{fn} (FN)'], [f'{fp} (FP)', f'{tn} (TN)']],
    index=pd.Index(CLASS_NAMES, name='Actual'),
    columns=pd.Index(CLASS_NAMES, name='Predicted'),
))

Predicted,bad,good
Actual,,
bad,62 (TP),28 (FN)
good,46 (FP),164 (TN)


In [14]:
print('Classification report')
print(classification_report(
    y_test,
    y_pred,
    labels=[BAD, GOOD],
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
))

Classification report
              precision    recall  f1-score   support

         bad     0.5741    0.6889    0.6263        90
        good     0.8542    0.7810    0.8159       210

    accuracy                         0.7533       300
   macro avg     0.7141    0.7349    0.7211       300
weighted avg     0.7701    0.7533    0.7590       300

